In [ ]:
# ================================================================
#  01_clean_qa — Normalize dùng chung (human + AI) + QA gate G2/G3
#  Input : human_samples_2018_clean.jsonl (get-dataset.ipynb)
#          human_ai_multi_free_models_v2.jsonl (genai.ipynb)
#  Output: human_clean.parquet, ai_clean.parquet
# ================================================================

import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd

# ── CELL 1: Config ──────────────────────────────────────────────
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path(".").resolve()

# Chay local: du lieu nam trong folder results/ (get-dataset.ipynb va genai.ipynb
# deu ghi/doc tu day, xem RESULTS_DIR trong genai.ipynb).
RESULTS_DIR = BASE_DIR / "results"

HUMAN_INPUT_PATH = RESULTS_DIR / "human_samples_2018_clean.jsonl"
AI_INPUT_PATH = RESULTS_DIR / "human_ai_multi_free_models_v2.jsonl"

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dung fastparquet thay vi pyarrow (mac dinh cua pandas) de ghi parquet: VS Code
# Jupyter extension dung pyarrow rieng cho preview DataFrame trong Variables pane,
# gay xung dot dang ky extension type toan cuc voi pyarrow cua chinh notebook
# (ArrowKeyError: "already defined" / "not found" cho pandas.period). fastparquet
# khong dung registry do cua pyarrow nen tranh duoc xung dot nay hoan toan.
PARQUET_ENGINE = "fastparquet"

MIN_WORDS = 100

print(f"BASE_DIR        : {BASE_DIR}")
print(f"HUMAN_INPUT_PATH : {'✅' if HUMAN_INPUT_PATH.exists() else '❌ KHÔNG TÌM THẤY'}  {HUMAN_INPUT_PATH}")
print(f"AI_INPUT_PATH    : {'✅' if AI_INPUT_PATH.exists() else '❌ KHÔNG TÌM THẤY'}  {AI_INPUT_PATH}")


# ── CELL 2: Normalize dùng chung (spec §7) ───────────────────────
# Áp CÙNG hàm này cho cả human và AI, đúng thứ tự: NFKC -> fold dấu câu
# typographic -> bỏ trích dẫn inline -> squeeze whitespace -> cắt preamble
# -> cắt code-fence/</think> -> lọc <100 từ -> loại refusal.

# Bản đồ dấu câu typographic -> ASCII (spec §7 bước 4: chính sách ký hiệu THỐNG
# NHẤT cho cả 2 lớp). NFKC KHÔNG fold smart-quote / en–em dash / minus (chỉ fold
# ellipsis), nên nếu bỏ qua thì các ký tự này thành "vân tay" của LLM: 39% mẫu AI
# có, 0% human ≤2018 (nguồn LaTeX/ASCII) -> shortcut bề mặt mà diagnostics §12 bắt.
_PUNCT_FOLD = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'",  # ‘ ’ ‚ ‛
    "“": '"', "”": '"', "„": '"', "‟": '"',  # “ ” „ ‟
    "′": "'", "″": '"',                                  # ′ ″ prime
    "‒": "-", "–": "-", "—": "-", "―": "-",  # ‒ – — ―
    "−": "-",                                                 # − minus
    "…": "...",                                              # … (NFKC cũng fold, để cho chắc)
})

# Trích dẫn inline là confound ĐỊNH DẠNG chứ không phải "ai viết chữ": human ≤2018
# giữ "(Author, 2018)" / "[12]" còn AI viết lại thì bỏ -> chênh punct/digit density
# giữa 2 lớp. Bỏ ở CẢ 2 lớp (shared) để khử confound (spec §0, §7).
_CITATION_RE = re.compile(r"\((?:[^()]*?\b\d{4}[a-z]?\b[^()]*?)\)")     # (Author, 2018) / (e.g., Smith 2016)
_BRACKET_REF_RE = re.compile(r"\[\s*\d{1,3}(?:\s*[,\-–]\s*\d{1,3})*\s*\]")  # [12], [3-5], [1, 2]

_PREAMBLE_RE = re.compile(
    r"^\s*(sure,?\s*)?here('?s| is)\s+(the\s+)?(rewritten|paraphrased|continu\w*|"
    r"expanded|revised|corrected)?\s*(excerpt|version|passage|paragraph|text)?\s*[:\-]\s*",
    flags=re.IGNORECASE,
)

_REFUSAL_PATTERNS = [
    re.compile(r"^\s*i\s+cannot\b", flags=re.IGNORECASE),
    re.compile(r"^\s*i'?m\s+sorry\b", flags=re.IGNORECASE),
    re.compile(r"^\s*as\s+an\s+ai\b", flags=re.IGNORECASE),
    re.compile(r"^\s*i\s+am\s+an\s+ai\b", flags=re.IGNORECASE),
    re.compile(r"^\s*i'?m\s+not\s+able\s+to\b", flags=re.IGNORECASE),
]


def normalize(text):
    if text is None:
        return ""

    text = str(text)

    # 1. Unicode NFKC
    text = unicodedata.normalize("NFKC", text)

    # 1b. Fold dấu câu typographic + bỏ trích dẫn inline (spec §7 bước 4 — khử
    #     shortcut dấu câu/trích dẫn ở diagnostics §12). Áp Y HỆT cho human & AI.
    text = text.translate(_PUNCT_FOLD)
    text = _CITATION_RE.sub(" ", text)
    text = _BRACKET_REF_RE.sub(" ", text)

    # 2. Chuẩn hoá khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    # 3. Cắt preamble ngôn ngữ tự nhiên của LLM trong ~250 ký tự đầu
    head = text[:250]
    m = _PREAMBLE_RE.match(head)

    if m:
        text = text[m.end():].strip()

    # + cắt code-fence / </think> còn sót (lớp bảo hiểm, dù genai.ipynb đã cắt)
    if "</think>" in text:
        text = text.split("</think>", 1)[-1].strip()

    if text.startswith("```") and text.endswith("```"):
        lines = text.splitlines()

        if len(lines) >= 3:
            text = "\n".join(lines[1:-1]).strip()

    return text


def is_refusal(text):
    head = text[:120]
    return any(p.match(head) for p in _REFUSAL_PATTERNS)


def has_leftover_preamble(text):
    return bool(_PREAMBLE_RE.match(text[:250]))


# ── CELL 3: Load + normalize HUMAN ────────────────────────────────
print("\n" + "=" * 60)
print("HUMAN")
print("=" * 60)

human_df = pd.read_json(HUMAN_INPUT_PATH, lines=True, dtype={"paper_id": "string"})
print(f"Raw human rows: {len(human_df):,}")

human_df["text"] = human_df["text"].apply(normalize)
human_df["word_count"] = human_df["text"].str.split().str.len()

before = len(human_df)
human_df = human_df[human_df["word_count"] >= MIN_WORDS].copy()
print(f"Lọc <{MIN_WORDS} từ: {before:,} -> {len(human_df):,}")

before = len(human_df)
human_df = human_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Dedup theo text (normalize): {before:,} -> {len(human_df):,}")

KEEP_HUMAN_COLS = [
    "paper_id", "source", "title", "categories", "published_year",
    "text", "word_count", "label",
]
human_df = human_df[[c for c in KEEP_HUMAN_COLS if c in human_df.columns]]


# ── CELL 4: Load + normalize AI (bao gồm cả ai_polish) ────────────
print("\n" + "=" * 60)
print("AI")
print("=" * 60)

ai_df = pd.read_json(AI_INPUT_PATH, lines=True, dtype={"paper_id": "string"})
print(f"Raw AI rows: {len(ai_df):,}")
print(ai_df["text_type"].value_counts(dropna=False).to_string())

# Chỉ giữ record thành công, đúng text_type cần cho pipeline (bỏ ai_failed)
is_success = ai_df["success"].astype(str).str.lower() == "true"
ai_df = ai_df[ai_df["text_type"].isin(["ai", "ai_polish"]) & is_success].copy()
print(f"Sau khi bỏ ai_failed / success=False: {len(ai_df):,}")

ai_df["text"] = ai_df["text"].apply(normalize)
ai_df["word_count"] = ai_df["text"].str.split().str.len()

before = len(ai_df)
ai_df = ai_df[ai_df["word_count"] >= MIN_WORDS].copy()
print(f"Lọc <{MIN_WORDS} từ: {before:,} -> {len(ai_df):,}")

before = len(ai_df)
ai_df = ai_df[~ai_df["text"].apply(is_refusal)].copy()
print(f"Loại refusal (\"I cannot...\", \"As an AI...\"): {before:,} -> {len(ai_df):,}")

before = len(ai_df)
ai_df = ai_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Dedup theo text (normalize): {before:,} -> {len(ai_df):,}")

KEEP_AI_COLS = [
    "paper_id", "source", "text", "word_count", "text_type",
    "split_pool", "generator_family", "model_actual", "provider",
    "gen_mode", "prompt_id", "temperature",
]
ai_df = ai_df[[c for c in KEEP_AI_COLS if c in ai_df.columns]]


# ── CELL 5: QA gate G2 (preamble sót) + G3 (min length) ───────────
print("\n" + "=" * 60)
print("QA GATES")
print("=" * 60)

g2_human_bad = human_df[human_df["text"].apply(has_leftover_preamble)]
g2_ai_bad = ai_df[ai_df["text"].apply(has_leftover_preamble)]
g2_pass = len(g2_human_bad) == 0 and len(g2_ai_bad) == 0

print(f"G2 (preamble sót = 0)      : {'✅ PASS' if g2_pass else '❌ FAIL'}"
      f"  (human={len(g2_human_bad)}, ai={len(g2_ai_bad)})")

if not g2_pass:
    print("  -> sample_id/paper_id còn preamble:")
    if len(g2_human_bad):
        print("     human:", g2_human_bad[["paper_id", "source"]].head(10).values.tolist())
    if len(g2_ai_bad):
        print("     ai   :", g2_ai_bad[["paper_id", "source"]].head(10).values.tolist())

g3_human_bad = (human_df["word_count"] < MIN_WORDS).sum()
g3_ai_bad = (ai_df["word_count"] < MIN_WORDS).sum()
g3_pass = g3_human_bad == 0 and g3_ai_bad == 0

print(f"G3 (mọi text >= {MIN_WORDS} từ) : {'✅ PASS' if g3_pass else '❌ FAIL'}"
      f"  (human vi phạm={g3_human_bad}, ai vi phạm={g3_ai_bad})")

if not (g2_pass and g3_pass):
    raise SystemExit(
        "❌ QA gate G2/G3 FAIL — dừng lại, không ghi output. "
        "Xem sample_id/paper_id in ở trên để sửa normalize() rồi chạy lại."
    )


# ── CELL 6: Ghi output ─────────────────────────────────────────────
human_out = OUTPUT_DIR / "human_clean.parquet"
ai_out = OUTPUT_DIR / "ai_clean.parquet"

human_df.to_parquet(human_out, index=False, engine=PARQUET_ENGINE)
ai_df.to_parquet(ai_out, index=False, engine=PARQUET_ENGINE)

print("\n" + "=" * 60)
print(f"✅ human_clean.parquet : {len(human_df):,} dòng -> {human_out}")
print(f"✅ ai_clean.parquet    : {len(ai_df):,} dòng -> {ai_out}")
print("=" * 60)
